# Forecasting Airline Stock Volatility with Oil Futures Volatility and Oil News Sentiment
## CS 329E — Phase 3: Implementing Design Plan
**Group 23** | Blake Stanley, Shivsagar Palla, Raghuvendra Chowdhry

**Date:** February 27, 2026

## 0: Project Goal

The goal of this project is to investigate whether volatility in crude oil futures — measured by the CBOE Crude Oil Volatility Index (OVX) — and oil market news sentiment — captured by the Text Oil Sentiment Indicator (TOSI) — can predict stock price volatility for major U.S. airlines (AAL, DAL, UAL, LUV) and the JETS ETF (a global airline industry index). We hypothesize that increases in OVX and negative shifts in TOSI lead to increased airline stock volatility, with the strongest predictive power at a 1-month lag. Combining OVX with TOSI should yield a more accurate model than either alone.

## 1. Setup

Import the required packages, define shared colors, and configure plotting defaults used throughout the notebook.

In [1]:
# --- Imports ---
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')  # suppress sklearn/pandas deprecation noise

import numpy as np
import pandas as pd
from IPython.display import display
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import explained_variance_score, mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb

pio.renderers.default = 'notebook_connected'  # render Plotly charts inline
pd.options.display.float_format = '{:,.3f}'.format  # 3 decimal places in all tables

# --- Constants ---
DATA_DIR = Path('data')
TICKERS = ['AAL', 'DAL', 'UAL', 'LUV', 'JETS']

# Brand color for each airline used consistently across all charts
AIRLINE_COLORS = {
    'AAL': '#C75146',
    'DAL': '#0B6E4F',
    'UAL': '#3C91E6',
    'LUV': '#F4A259',
    'JETS': '#7A5195',
}

# One color per model+specification combination for comparison bar charts
MODEL_COLORS = {
    'MIDAS | OVX Only': '#8C564B',
    'MIDAS | TOSI Only': '#4C78A8',
    'MIDAS | OVX + TOSI': '#2D3047',
    'XGBoost | OVX Only': '#E45756',
    'XGBoost | TOSI Only': '#72B7B2',
    'XGBoost | OVX + TOSI': '#1B998B',
    'Random Forest | OVX Only': '#F58518',
    'Random Forest | TOSI Only': '#FF9DA6',
    'Random Forest | OVX + TOSI': '#FF6B35',
    'OVX Only': '#C75146',
    'TOSI Only': '#3C91E6',
    'OVX + TOSI': '#0B6E4F',
}

# Date windows for major market shocks — rendered as grey bands on time-series charts
CRISIS_PERIODS = [
    ('2008-09-01', '2009-06-30', 'Financial crisis'),
    ('2020-02-01', '2020-07-31', 'COVID travel shock'),
    ('2022-02-01', '2022-08-31', 'Energy shock'),
]

# Shared Plotly theme applied to every figure through style_figure()
BASE_LAYOUT = dict(
    template='plotly_white',
    paper_bgcolor='#F7F4ED',
    plot_bgcolor='#FFFFFF',
    font=dict(family='Aptos, Segoe UI, sans-serif', size=13, color='#24323D'),
    colorway=['#0B6E4F', '#C75146', '#3C91E6', '#F4A259', '#7A5195', '#2D3047'],
    hoverlabel=dict(bgcolor='white', font_size=12),
    margin=dict(l=60, r=30, t=80, b=120),
)


def style_figure(fig, title, height=550, **_kw):
    # Apply shared layout, title, and legend position to any Plotly figure
    fig.update_layout(
        title=dict(text=title, x=0.02, xanchor='left'),
        height=height,
        legend=dict(orientation='h', y=-0.22, x=1, xanchor='right', yanchor='top', font=dict(size=11)),
        hovermode='x unified',
        **BASE_LAYOUT,
    )
    return fig

## 2. Metrics and Data Preparation

Define the model evaluation metrics and the functions that load, clean, and align the airline volatility, OVX, and TOSI datasets.

In [2]:
# --- Forecast evaluation metrics ---

def rmse(y_true, y_pred):
    # Root mean squared error
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def directional_accuracy(actual, predicted, previous):
    # Fraction of months where the model correctly predicts whether
    # volatility rose or fell relative to the prior month
    actual_direction    = np.sign(np.asarray(actual)    - np.asarray(previous))
    predicted_direction = np.sign(np.asarray(predicted) - np.asarray(previous))
    return float((actual_direction == predicted_direction).mean())


def evaluate_forecast(actual, predicted, previous):
    # Return a full metric dict for a set of out-of-sample predictions.
    # 'previous' is the prior-month volatility, needed for directional accuracy.
    actual    = np.asarray(actual,    dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    previous  = np.asarray(previous,  dtype=float)
    errors        = actual - predicted
    abs_actual    = np.where(np.abs(actual) < 1e-8, np.nan, np.abs(actual))
    smape_denom   = np.abs(actual) + np.abs(predicted)

    return {
        'RMSE':               rmse(actual, predicted),
        'MAE':                float(mean_absolute_error(actual, predicted)),
        'MedAE':              float(median_absolute_error(actual, predicted)),
        'MAPE':               float(np.nanmean(np.abs(errors) / abs_actual) * 100),
        'sMAPE':              float(np.nanmean(2 * np.abs(errors) / np.where(smape_denom < 1e-8, np.nan, smape_denom)) * 100),
        'Bias':               float(np.mean(predicted - actual)),  # positive = model over-predicts on average
        'Explained_Variance': float(explained_variance_score(actual, predicted)) if len(actual) > 1 else np.nan,
        'R2':                 float(r2_score(actual, predicted))   if len(actual) > 1 else np.nan,
        'Directional_Accuracy': directional_accuracy(actual, predicted, previous),
    }


def almon_weighted_average(values, theta1=-0.15, theta2=-0.01):
    # Collapse a window of high-frequency values into a single scalar using
    # Almon (polynomial) lag weights. Earlier lags receive less weight when
    # theta1 < 0.  This is the core aggregation step in MIDAS regression.
    values = np.asarray(values, dtype=float)
    lags   = np.arange(len(values), dtype=float)
    raw     = np.exp(theta1 * lags + theta2 * (lags ** 2))
    weights = raw / raw.sum()                  # normalize so weights sum to 1
    return float(np.dot(values[::-1], weights))  # most-recent obs gets lag-0 weight


# Grid of Almon shape parameters explored during hyperparameter tuning
THETA1_GRID = [-0.35, -0.20, -0.10, -0.05, 0.00]
THETA2_GRID = [0.00, -0.005, -0.010, -0.020]


def polynomial_fit_scores(x, y):
    # Fit linear, quadratic, and cubic polynomials to (x, y).
    # Returns R² for each degree and the label of the best-fitting curve.
    scores = {}
    for degree, label in [(1, 'Linear'), (2, 'Quadratic'), (3, 'Cubic')]:
        coeffs = np.polyfit(x, y, degree)
        poly   = np.poly1d(coeffs)
        pred   = poly(x)
        ss_res = np.sum((y - pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        scores[f'R2_{label}'] = np.nan if ss_tot == 0 else 1 - ss_res / ss_tot
    scores['Best_Fit'] = max(['Linear', 'Quadratic', 'Cubic'], key=lambda name: scores[f'R2_{name}'])
    return scores


# --- Data loading and cleaning ---

def load_project_data():
    # Load all raw datasets, clean them, and build daily and monthly modeling panels.
    # Returns iv_data, ovx, tosi, daily_panel, monthly_panel, data_inventory, cleaning_log.
    iv_data        = {}
    inventory_rows = []
    cleaning_rows  = []

    # Load each airline's implied-volatility Excel file (Bloomberg export, skiprows=6 skips the header block)
    for ticker in TICKERS:
        raw = pd.read_excel(DATA_DIR / f'{ticker}_IV.xlsx', skiprows=6)
        raw['Date'] = pd.to_datetime(raw['Date'])
        col = f'{ticker}_Volatility'
        raw = raw.rename(columns={'HIST_CALL_IMP_VOL': col})[['Date', col]].sort_values('Date').reset_index(drop=True)
        missing_before = int(raw[col].isna().sum())
        raw_rows = len(raw)

        # Backfill isolated NaN gaps, then drop any remaining leading NaNs
        clean = raw.copy()
        clean[col] = clean[col].bfill()
        clean = clean.dropna(subset=[col]).reset_index(drop=True)
        iv_data[ticker] = clean

        inventory_rows.append({
            'Dataset': f'{ticker} IV', 'Rows': raw_rows, 'Columns': 1,
            'Start': raw['Date'].min().date(), 'End': raw['Date'].max().date(),
            'Missing_Before_Cleaning': missing_before,
        })
        cleaning_rows.append({
            'Dataset': f'{ticker} IV',
            'Method': 'Backfill then drop leading structural gaps',
            'Rows_Removed': raw_rows - len(clean),
            'Missing_Before': missing_before,
            'Missing_After': int(clean[col].isna().sum()),
        })

    # Load OVX (CBOE crude oil volatility index); drop holiday rows that have no published value
    ovx_raw = pd.read_csv(DATA_DIR / 'OVXCLS.csv')
    ovx_raw['observation_date'] = pd.to_datetime(ovx_raw['observation_date'])
    ovx_raw = ovx_raw.rename(columns={'observation_date': 'Date', 'OVXCLS': 'OVX'})
    ovx_raw['OVX'] = pd.to_numeric(ovx_raw['OVX'], errors='coerce')
    ovx_missing = int(ovx_raw['OVX'].isna().sum())
    ovx = ovx_raw.dropna(subset=['OVX']).sort_values('Date').reset_index(drop=True)

    inventory_rows.append({
        'Dataset': 'OVX', 'Rows': len(ovx_raw), 'Columns': 1,
        'Start': ovx_raw['Date'].min().date(), 'End': ovx_raw['Date'].max().date(),
        'Missing_Before_Cleaning': ovx_missing,
    })
    cleaning_rows.append({
        'Dataset': 'OVX',
        'Method': 'Drop holiday rows with no published index value',
        'Rows_Removed': len(ovx_raw) - len(ovx),
        'Missing_Before': ovx_missing, 'Missing_After': int(ovx['OVX'].isna().sum()),
    })

    # Load TOSI (monthly oil-news sentiment index); no cleaning required
    tosi = pd.read_csv(DATA_DIR / 'TOSI.csv', usecols=['Date', 'TOSI'])
    tosi['Date'] = pd.to_datetime(tosi['Date'], format='%b-%y')
    tosi = tosi.sort_values('Date').reset_index(drop=True)

    inventory_rows.append({
        'Dataset': 'TOSI', 'Rows': len(tosi), 'Columns': 1,
        'Start': tosi['Date'].min().date(), 'End': tosi['Date'].max().date(),
        'Missing_Before_Cleaning': int(tosi['TOSI'].isna().sum()),
    })
    cleaning_rows.append({
        'Dataset': 'TOSI', 'Method': 'No cleaning needed', 'Rows_Removed': 0,
        'Missing_Before': int(tosi['TOSI'].isna().sum()),
        'Missing_After':  int(tosi['TOSI'].isna().sum()),
    })

    # Build a daily panel: OVX is the spine; airline IV series are left-joined by date.
    # TOSI is monthly so it is forward-filled to cover every trading day.
    daily_panel = ovx.set_index('Date')[['OVX']].copy()
    for ticker in TICKERS:
        col = f'{ticker}_Volatility'
        daily_panel = daily_panel.join(iv_data[ticker].set_index('Date')[[col]], how='left')
    daily_panel = daily_panel.join(tosi.set_index('Date')[['TOSI']], how='left')
    daily_panel['TOSI'] = daily_panel['TOSI'].ffill()
    daily_panel = daily_panel.sort_index()

    # Build a monthly panel by resampling IV to month-start means, then joining
    # OVX summary stats (mean, max, min, std) and TOSI for ML feature engineering
    monthly_iv = [
        iv_data[ticker].set_index('Date')[[f'{ticker}_Volatility']].resample('MS').mean()
        for ticker in TICKERS
    ]
    monthly_panel = pd.concat(monthly_iv, axis=1)
    ovx_monthly = ovx.set_index('Date').resample('MS').agg(
        OVX_mean=('OVX', 'mean'),
        OVX_max=('OVX', 'max'),
        OVX_min=('OVX', 'min'),
        OVX_std=('OVX', 'std'),
    )
    monthly_panel = monthly_panel.join(ovx_monthly, how='outer')
    monthly_panel = monthly_panel.join(tosi.set_index('Date')[['TOSI']], how='outer').sort_index()

    return (
        iv_data, ovx, tosi, daily_panel, monthly_panel,
        pd.DataFrame(inventory_rows), pd.DataFrame(cleaning_rows),
    )


# Load all datasets and build the daily and monthly modeling panels
iv_data, ovx, tosi, daily_panel, monthly_panel, data_inventory, cleaning_log = load_project_data()

# Columns used for the monthly correlation matrix (all IVs + OVX mean + TOSI)
monthly_corr_cols = [f'{ticker}_Volatility' for ticker in TICKERS] + ['OVX_mean', 'TOSI']
monthly_corr = monthly_panel[monthly_corr_cols].corr()
full_overlap  = monthly_panel[monthly_corr_cols].dropna()  # rows where every series has data

# For each (driver, airline) pair, fit linear/quadratic/cubic curves and record
# R² for each degree — used to justify non-linear models downstream
fit_rows = []
for ticker in TICKERS:
    vol_col = f'{ticker}_Volatility'

    ovx_pair  = monthly_panel[['OVX_mean', vol_col]].dropna()
    ovx_stats = polynomial_fit_scores(ovx_pair['OVX_mean'].to_numpy(), ovx_pair[vol_col].to_numpy())
    fit_rows.append({
        'Pair': f'OVX vs {ticker}',
        'Correlation': ovx_pair['OVX_mean'].corr(ovx_pair[vol_col]),
        **ovx_stats,
    })

    tosi_pair  = monthly_panel[['TOSI', vol_col]].dropna()
    tosi_stats = polynomial_fit_scores(tosi_pair['TOSI'].to_numpy(), tosi_pair[vol_col].to_numpy())
    fit_rows.append({
        'Pair': f'TOSI vs {ticker}',
        'Correlation': tosi_pair['TOSI'].corr(tosi_pair[vol_col]),
        **tosi_stats,
    })

fit_summary = pd.DataFrame(fit_rows)

print(f'Daily modeling panel: {daily_panel.shape[0]:,} trading days from {daily_panel.index.min().date()} to {daily_panel.index.max().date()}')
print(f'Monthly full-overlap panel: {full_overlap.shape[0]} months from {full_overlap.index.min().date()} to {full_overlap.index.max().date()}')

## 3. Modeling Utilities

Create helper functions for MIDAS windows, monthly machine learning features, and time-series cross-validation. These utilities build the inputs for the forecasting models.

In [3]:
# --- MIDAS window construction ---

def build_midas_windows(monthly_target, ovx_daily, tosi_monthly, horizon_months, trading_days_per_month=21):
    # For each month t, collect the trailing OVX daily window and TOSI monthly window
    # that will be used to predict month t+1 volatility.
    # Rows with insufficient history (< 15 trading days per month) are skipped.
    monthly_target = monthly_target.dropna().sort_index()
    rows = []

    for feature_month in monthly_target.index:
        target_month = feature_month + pd.offsets.MonthBegin(1)
        if target_month not in monthly_target.index:
            continue  # skip if the realized target month is not yet available

        month_end   = feature_month + pd.offsets.MonthEnd(0)
        ovx_window  = ovx_daily.loc[:month_end].tail(horizon_months * trading_days_per_month)
        tosi_window = tosi_monthly.loc[:feature_month].tail(horizon_months)

        if len(ovx_window) < horizon_months * 15 or len(tosi_window) < horizon_months:
            continue

        rows.append({
            'feature_month':    feature_month,
            'target_date':      target_month,
            'previous_vol':     float(monthly_target.loc[feature_month]),
            'target_next_month': float(monthly_target.loc[target_month]),
            'ovx_window':       ovx_window.to_numpy(dtype=float),
            'tosi_window':      tosi_window.to_numpy(dtype=float),
        })

    return pd.DataFrame(rows).set_index('feature_month')


def build_midas_design(window_df, spec, theta_params):
    # Convert raw OVX/TOSI windows into Almon-weighted scalar features.
    # 'spec' controls which drivers are included: 'OVX Only', 'TOSI Only', or 'OVX + TOSI'.
    design = pd.DataFrame(index=window_df.index)
    if 'OVX' in spec:
        theta1, theta2 = theta_params['ovx']
        design['OVX_midas']  = window_df['ovx_window'].apply(lambda arr: almon_weighted_average(arr, theta1, theta2))
    if 'TOSI' in spec:
        theta1, theta2 = theta_params['tosi']
        design['TOSI_midas'] = window_df['tosi_window'].apply(lambda arr: almon_weighted_average(arr, theta1, theta2))
    return design


def tune_midas_weights(window_df, spec):
    # Grid-search Almon theta parameters on a held-out validation slice taken
    # from the first 75% of data, before the out-of-sample test period.
    # Returns the (theta1, theta2) pair that minimizes validation RMSE.
    train_cut = max(24, int(np.ceil(len(window_df) * 0.75)))
    tune_cut  = max(18, train_cut - 12)
    tune_train = window_df.iloc[:tune_cut]
    tune_val   = window_df.iloc[tune_cut:train_cut]
    if tune_val.empty:
        tune_train = window_df.iloc[:-6]
        tune_val   = window_df.iloc[-6:]

    # Full Cartesian product of candidate theta values
    candidate_pairs = [(t1, t2) for t1 in THETA1_GRID for t2 in THETA2_GRID]
    ovx_pairs  = candidate_pairs if 'OVX'  in spec else [(None, None)]
    tosi_pairs = candidate_pairs if 'TOSI' in spec else [(None, None)]

    best_score  = np.inf
    best_params = {}

    for ovx_pair in ovx_pairs:
        for tosi_pair in tosi_pairs:
            theta_params = {}
            if 'OVX'  in spec: theta_params['ovx']  = ovx_pair
            if 'TOSI' in spec: theta_params['tosi'] = tosi_pair

            X_train = build_midas_design(tune_train, spec, theta_params)
            X_val   = build_midas_design(tune_val,   spec, theta_params)
            model   = LinearRegression().fit(X_train, tune_train['target_next_month'])
            pred    = model.predict(X_val)
            score   = rmse(tune_val['target_next_month'], pred)

            if score < best_score:
                best_score  = score
                best_params = theta_params.copy()

    return best_params


def rolling_midas_forecast(window_df, spec, min_train=36):
    # Produce rolling one-step-ahead MIDAS forecasts using an expanding window.
    # The first 70% of observations form the initial training set; theta weights
    # are tuned once on that set and then held fixed for all forecast steps.
    if len(window_df) <= min_train:
        return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan, 'Directional_Accuracy': np.nan}, pd.DataFrame(), {}

    start_test   = max(min_train, int(np.ceil(len(window_df) * 0.70)))
    theta_params = tune_midas_weights(window_df.iloc[:start_test], spec)
    prediction_rows = []

    # Expanding-window loop: each step trains on all months up to end_idx
    for end_idx in range(start_test, len(window_df)):
        train = window_df.iloc[:end_idx]
        test  = window_df.iloc[[end_idx]]

        X_train = build_midas_design(train, spec, theta_params)
        X_test  = build_midas_design(test,  spec, theta_params)
        model   = LinearRegression().fit(X_train, train['target_next_month'])
        pred    = float(model.predict(X_test)[0])

        prediction_rows.append({
            'target_date':  test['target_date'].iloc[0],
            'previous_vol': float(test['previous_vol'].iloc[0]),
            'actual':       float(test['target_next_month'].iloc[0]),
            'predicted':    pred,
        })

    prediction_df = pd.DataFrame(prediction_rows)
    metrics = evaluate_forecast(prediction_df['actual'], prediction_df['predicted'], prediction_df['previous_vol'])
    return metrics, prediction_df, theta_params


# --- Monthly ML feature engineering ---

def build_monthly_ml_frame(monthly_panel, ticker, feature_spec='OVX + TOSI', lags=(1, 3, 6)):
    # Build a lagged feature matrix for XGBoost / Random Forest.
    # For each lag (1, 3, 6 months), creates lagged columns of OVX stats, TOSI,
    # and the target ticker's own past volatility.  Also adds an OVX intra-month
    # range feature.  Returns the feature DataFrame and active feature column names.
    target_col = f'{ticker}_Volatility'
    base_cols  = [target_col, 'OVX_mean', 'OVX_max', 'OVX_min', 'OVX_std', 'TOSI']
    frame      = monthly_panel[base_cols].dropna().copy()
    feature_cols = []

    include_ovx  = feature_spec in ['OVX Only',  'OVX + TOSI']
    include_tosi = feature_spec in ['TOSI Only', 'OVX + TOSI']

    for lag in lags:
        for col in ['OVX_mean', 'OVX_max', 'OVX_min', 'OVX_std', 'TOSI', target_col]:
            feature_name     = f'{col}_lag{lag}m'
            frame[feature_name] = frame[col].shift(lag)  # shift creates the look-back feature
            if   col.startswith('OVX') and include_ovx:  feature_cols.append(feature_name)
            elif col == 'TOSI'         and include_tosi: feature_cols.append(feature_name)
            elif col == target_col:                       feature_cols.append(feature_name)  # own lags always included

    # High-low OVX range captures intra-month crude oil stress
    frame['OVX_range_lag1m'] = frame['OVX_max'].shift(1) - frame['OVX_min'].shift(1)
    if include_ovx:
        feature_cols.append('OVX_range_lag1m')

    # Target is next month's volatility; previous_vol is used for directional accuracy
    frame['target_next_month'] = frame[target_col].shift(-1)
    frame['previous_vol']      = frame[target_col]
    frame['target_date']       = frame.index + pd.offsets.MonthBegin(1)
    frame = frame.dropna(subset=feature_cols + ['target_next_month'])
    return frame, feature_cols


# --- Model factories ---

def make_xgb_model():
    # Lightly regularized XGBoost; shallow trees (max_depth=3) reduce overfitting
    # on the small monthly dataset (~100 training rows)
    return xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=400,
        max_depth=3,
        learning_rate=0.04,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=4,
        reg_lambda=1.25,
        random_state=42,
    )


def make_rf_model():
    # Random Forest with depth-limited trees to prevent overfitting
    return RandomForestRegressor(
        n_estimators=500,
        max_depth=6,
        min_samples_leaf=3,
        random_state=42,
    )


def time_series_cv(df_ml, feature_cols, spec_label='OVX + TOSI'):
    # Run TimeSeriesSplit cross-validation on the training portion (first 80%) of data.
    # Returns per-fold RMSE for XGBoost and Random Forest to assess out-of-sample stability.
    split_idx = int(len(df_ml) * 0.80)
    X_train   = df_ml[feature_cols].iloc[:split_idx]
    y_train   = df_ml['target_next_month'].iloc[:split_idx]
    n_splits  = min(5, max(2, len(X_train) // 12))  # roughly one fold per year
    splitter  = TimeSeriesSplit(n_splits=n_splits)

    rows = []
    for model_name, factory in [('XGBoost', make_xgb_model), ('Random Forest', make_rf_model)]:
        for fold, (train_idx, val_idx) in enumerate(splitter.split(X_train), start=1):
            model = factory()
            model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
            pred  = model.predict(X_train.iloc[val_idx])
            rows.append({
                'Model': model_name, 'Specification': spec_label,
                'Fold': fold, 'RMSE': rmse(y_train.iloc[val_idx], pred),
            })
    return pd.DataFrame(rows)

Daily modeling panel: 4,731 trading days from 2007-05-10 to 2026-02-25
Monthly full-overlap panel: 127 months from 2015-04-01 to 2025-10-01


## 4. Train and Evaluate Models

Fit each model family under three specifications: **OVX only**, **TOSI only**, and **OVX + TOSI**. MIDAS is evaluated with rolling out-of-sample forecasts, while XGBoost and Random Forest use an 80/20 temporal train/test split.

In [4]:
# Index series used by the MIDAS builder (daily OVX spine, monthly TOSI spine)
ovx_daily    = ovx.set_index('Date')['OVX']
tosi_monthly = tosi.set_index('Date')['TOSI']

SPECS         = ['OVX Only', 'TOSI Only', 'OVX + TOSI']
MODEL_FAMILIES = ['MIDAS', 'XGBoost', 'Random Forest']

# --- MIDAS: horizon sensitivity (JETS only) ---
# Test 1-, 3-, and 6-month OVX windows on JETS to find the most informative lag length
midas_horizon_rows   = []
midas_jets_predictions = {}

for horizon in [1, 3, 6]:
    jets_windows = build_midas_windows(monthly_panel['JETS_Volatility'], ovx_daily, tosi_monthly, horizon)
    for spec in SPECS:
        metrics, pred_df, theta_params = rolling_midas_forecast(jets_windows, spec)
        midas_horizon_rows.append({
            'Horizon_Months': horizon, 'Horizon_Label': f'{horizon}-month',
            'Specification': spec,
            'RMSE': metrics['RMSE'], 'MAE': metrics['MAE'], 'MedAE': metrics['MedAE'],
            'MAPE': metrics['MAPE'], 'sMAPE': metrics['sMAPE'], 'Bias': metrics['Bias'],
            'Explained_Variance': metrics['Explained_Variance'], 'R2': metrics['R2'],
            'Directional_Accuracy': metrics['Directional_Accuracy'],
            'Test_Months': len(pred_df),
            'OVX_Theta': theta_params.get('ovx'), 'TOSI_Theta': theta_params.get('tosi'),
        })
        if spec == 'OVX + TOSI':
            midas_jets_predictions[horizon] = pred_df.copy()

midas_horizon_df = pd.DataFrame(midas_horizon_rows).sort_values(['Horizon_Months', 'Specification']).reset_index(drop=True)

# --- MIDAS: all tickers, 1-month horizon ---
# Rolling out-of-sample forecasts for every airline under all three feature specifications
midas_rows       = []
midas_predictions = {}

for ticker in TICKERS:
    windows = build_midas_windows(monthly_panel[f'{ticker}_Volatility'], ovx_daily, tosi_monthly, 1)
    for spec in SPECS:
        metrics, pred_df, theta_params = rolling_midas_forecast(windows, spec)
        model_label = f'MIDAS | {spec}'
        if spec == 'OVX + TOSI':
            midas_predictions[ticker] = pred_df.copy()  # save for the forecast comparison chart
        midas_rows.append({
            'Ticker': ticker, 'Family': 'MIDAS', 'Specification': spec, 'Model': model_label,
            'RMSE': metrics['RMSE'], 'MAE': metrics['MAE'], 'MedAE': metrics['MedAE'],
            'MAPE': metrics['MAPE'], 'sMAPE': metrics['sMAPE'], 'Bias': metrics['Bias'],
            'Explained_Variance': metrics['Explained_Variance'], 'R2': metrics['R2'],
            'Directional_Accuracy': metrics['Directional_Accuracy'],
            'Test_Months': len(pred_df),
            'OVX_Theta': theta_params.get('ovx'), 'TOSI_Theta': theta_params.get('tosi'),
        })

midas_all_df = pd.DataFrame(midas_rows)

# --- XGBoost and Random Forest: all tickers, 80/20 temporal split ---
# Train on the first 80% of monthly observations; evaluate on the final 20%
ml_rows           = []
ml_predictions    = {}
feature_importances = {}

for ticker in TICKERS:
    combined_prediction_frame = None
    for spec in SPECS:
        df_ml, feature_cols = build_monthly_ml_frame(monthly_panel, ticker, feature_spec=spec)
        split_idx = int(len(df_ml) * 0.80)

        X_train       = df_ml[feature_cols].iloc[:split_idx]
        X_test        = df_ml[feature_cols].iloc[split_idx:]
        y_train       = df_ml['target_next_month'].iloc[:split_idx]
        y_test        = df_ml['target_next_month'].iloc[split_idx:]
        previous_test = df_ml['previous_vol'].iloc[split_idx:]

        # Collect actual values and dates for the forecast comparison chart later
        prediction_frame = pd.DataFrame({
            'target_date':  df_ml['target_date'].iloc[split_idx:].to_numpy(),
            'actual':       y_test.to_numpy(),
            'previous_vol': previous_test.to_numpy(),
        })

        for model_name, factory in [('XGBoost', make_xgb_model), ('Random Forest', make_rf_model)]:
            model = factory()
            model.fit(X_train, y_train)
            pred    = model.predict(X_test)
            metrics = evaluate_forecast(y_test, pred, previous_test)
            model_label = f'{model_name} | {spec}'

            ml_rows.append({
                'Ticker': ticker, 'Family': model_name, 'Specification': spec, 'Model': model_label,
                'RMSE': metrics['RMSE'], 'MAE': metrics['MAE'], 'MedAE': metrics['MedAE'],
                'MAPE': metrics['MAPE'], 'sMAPE': metrics['sMAPE'], 'Bias': metrics['Bias'],
                'Explained_Variance': metrics['Explained_Variance'], 'R2': metrics['R2'],
                'Directional_Accuracy': metrics['Directional_Accuracy'],
                'Train_Months': len(X_train), 'Test_Months': len(X_test),
            })

            if spec == 'OVX + TOSI':
                prediction_frame[model_name] = pred
                # Store feature importances for the combined spec to power the importance chart
                feature_importances.setdefault(ticker, {})[model_name] = pd.Series(
                    model.feature_importances_, index=feature_cols,
                ).sort_values(ascending=False)

        if spec == 'OVX + TOSI':
            combined_prediction_frame = prediction_frame.copy()

    ml_predictions[ticker] = combined_prediction_frame

ml_results_df = pd.DataFrame(ml_rows)

# --- Cross-validation on JETS ---
# Assess within-training stability via TimeSeriesSplit; high fold-to-fold variance
# indicates the model is sensitive to which time window it sees
cv_rows = []
for spec in SPECS:
    jets_ml_frame, jets_feature_cols = build_monthly_ml_frame(monthly_panel, 'JETS', feature_spec=spec)
    cv_rows.append(time_series_cv(jets_ml_frame, jets_feature_cols, spec_label=spec))
cv_results_df = pd.concat(cv_rows, ignore_index=True)
cv_summary_df = cv_results_df.groupby(['Model', 'Specification'], as_index=False).agg(
    Fold_RMSE_Mean=('RMSE', 'mean'),
    Fold_RMSE_Std=('RMSE', 'std'),
)

# --- Aggregate result tables ---

# Combine MIDAS and ML results into one unified comparison DataFrame
model_comparison_df = pd.concat([
    midas_all_df[['Ticker', 'Family', 'Specification', 'Model', 'RMSE', 'MAE', 'MedAE', 'MAPE', 'sMAPE', 'Bias', 'Explained_Variance', 'R2', 'Directional_Accuracy']],
    ml_results_df[['Ticker', 'Family', 'Specification', 'Model', 'RMSE', 'MAE', 'MedAE', 'MAPE', 'sMAPE', 'Bias', 'Explained_Variance', 'R2', 'Directional_Accuracy']],
], ignore_index=True)

# Pick the lowest-RMSE model for each ticker (used to annotate the forecast chart)
best_models_df = model_comparison_df.loc[
    model_comparison_df.groupby('Ticker')['RMSE'].idxmin()
].sort_values('Ticker').reset_index(drop=True)

# Average metrics by model family and feature specification
family_spec_summary_df = model_comparison_df.groupby(['Family', 'Specification'], as_index=False).agg(
    Avg_RMSE=('RMSE', 'mean'), Avg_MAE=('MAE', 'mean'),
    Avg_MAPE=('MAPE', 'mean'), Avg_R2=('R2', 'mean'),
    Avg_Directional_Accuracy=('Directional_Accuracy', 'mean'),
)

# Full metric snapshot sorted by ticker then RMSE (used in the results table)
metrics_snapshot_df = model_comparison_df[[
    'Ticker', 'Family', 'Specification', 'Model', 'RMSE', 'MAE', 'MedAE', 'MAPE', 'sMAPE',
    'Bias', 'Explained_Variance', 'R2', 'Directional_Accuracy'
]].sort_values(['Ticker', 'RMSE']).reset_index(drop=True)

# Average metrics across all tickers, ranked by RMSE — the headline leaderboard
aggregate_metrics_df = model_comparison_df.groupby('Model', as_index=False).agg(
    Avg_RMSE=('RMSE', 'mean'), Avg_MAE=('MAE', 'mean'), Avg_MedAE=('MedAE', 'mean'),
    Avg_MAPE=('MAPE', 'mean'), Avg_sMAPE=('sMAPE', 'mean'),
    Avg_Abs_Bias=('Bias', lambda s: np.mean(np.abs(s))),
    Avg_Explained_Variance=('Explained_Variance', 'mean'),
    Avg_R2=('R2', 'mean'),
    Avg_Directional_Accuracy=('Directional_Accuracy', 'mean'),
).sort_values('Avg_RMSE').reset_index(drop=True)

visual_catalog = pd.DataFrame([
    {'Visual': '1. Daily airline volatility lines', 'Interactive_Element': 'Range slider + hover + crisis shading', 'Annotated': 'Yes'},
    {'Visual': '2. OVX/TOSI dual-axis drivers', 'Interactive_Element': 'Unified hover + crisis callouts', 'Annotated': 'Yes'},
    {'Visual': '3. Monthly correlation heatmap', 'Interactive_Element': 'Hover correlation lookup', 'Annotated': 'Yes'},
    {'Visual': '4. OVX vs airline scatter facets', 'Interactive_Element': 'Hover + fitted cubic curves', 'Annotated': 'Yes'},
    {'Visual': '5. TOSI vs airline scatter facets', 'Interactive_Element': 'Hover + fitted trend lines', 'Annotated': 'Yes'},
    {'Visual': '6. Model/spec comparison bars', 'Interactive_Element': 'Hover metrics + best-model labels', 'Annotated': 'Yes'},
    {'Visual': '7. Forecast comparison panels', 'Interactive_Element': 'Legend filtering + hover', 'Annotated': 'Yes'},
    {'Visual': '8. MIDAS horizon comparison', 'Interactive_Element': 'Grouped interactive bars', 'Annotated': 'Yes'},
    {'Visual': '9. Feature importance panels', 'Interactive_Element': 'Hover-ranked features', 'Annotated': 'Yes'},
])
print('Modeling complete.')
print('Best RMSE by ticker:')
print(best_models_df[['Ticker', 'Model', 'RMSE']].to_string(index=False))

Modeling complete.
Best RMSE by ticker:
Ticker                     Model  RMSE
   AAL          MIDAS | OVX Only 3.803
   DAL       XGBoost | TOSI Only 6.073
  JETS Random Forest | TOSI Only 4.778
   LUV  Random Forest | OVX Only 6.500
   UAL  Random Forest | OVX Only 5.732


## 5. Interactive Visual Design

Build the Plotly figures used to explain the data, compare the full set of model specifications, and satisfy the project visualization requirements.

In [5]:
# --- Shared chart utilities ---

def add_crisis_bands(fig):
    # Shade each crisis period with a light grey band and a small label
    positions = ['top left', 'bottom left', 'top left']
    for (start, end, label), pos in zip(CRISIS_PERIODS, positions):
        fig.add_vrect(
            x0=start, x1=end,
            fillcolor='rgba(125, 125, 125, 0.10)', line_width=0,
            annotation_text=label, annotation_position=pos,
            annotation_font=dict(size=8, color='rgba(100,100,100,0.55)'),
        )
    return fig


def axis_suffix(position):
    # Plotly subplot axis suffix: '' for the first subplot, '2', '3', ... for the rest
    return '' if position == 1 else str(position)


# --- Figure builders ---

def make_airline_timeseries_figure():
    # Line chart of daily implied volatility for all five tickers, with crisis shading
    fig = go.Figure()
    for ticker in TICKERS:
        col = f'{ticker}_Volatility'
        df  = iv_data[ticker]
        fig.add_trace(go.Scatter(
            x=df['Date'], y=df[col],
            mode='lines', name=ticker,
            line=dict(color=AIRLINE_COLORS[ticker], width=2),
            hovertemplate=f'{ticker}<br>%{{x|%Y-%m-%d}}<br>IV=%{{y:.2f}}<extra></extra>',
        ))

    add_crisis_bands(fig)
    fig.update_xaxes(rangeslider_visible=True, title='Date')
    fig.update_yaxes(title='Implied Volatility (%)')
    fig.add_annotation(
        x='2020-03-16', y=95,
        text='COVID shock pushed sector volatility sharply higher',
        showarrow=True, arrowhead=2, bgcolor='rgba(255,255,255,0.85)',
    )
    return style_figure(fig, 'Daily Implied Volatility Across Major U.S. Airlines and JETS', height=620)


def make_oil_driver_figure():
    # Dual-axis chart: OVX (left axis) and TOSI (right axis) through time
    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Scatter(
        x=ovx['Date'], y=ovx['OVX'],
        mode='lines', name='OVX',
        line=dict(color='#C75146', width=2),
        hovertemplate='OVX<br>%{x|%Y-%m-%d}<br>%{y:.2f}<extra></extra>',
    ), secondary_y=False)
    fig.add_trace(go.Scatter(
        x=tosi['Date'], y=tosi['TOSI'],
        mode='lines+markers', name='TOSI',
        line=dict(color='#3C91E6', width=2.5), marker=dict(size=5),
        hovertemplate='TOSI<br>%{x|%Y-%m}<br>%{y:.2f}<extra></extra>',
    ), secondary_y=True)

    add_crisis_bands(fig)
    fig.add_hline(y=0, line_dash='dash', line_color='#7F8C8D', secondary_y=True)  # neutral sentiment line
    fig.update_xaxes(title='Date')
    fig.update_yaxes(title='OVX Index Level', secondary_y=False)
    fig.update_yaxes(title='TOSI Sentiment Index', secondary_y=True)
    fig.add_annotation(
        x=0.99, y=0.99, xref='paper', yref='paper', xanchor='right', yanchor='top',
        text='Oil-market shocks simultaneously raised OVX and weakened sentiment',
        showarrow=False, font=dict(size=11), bgcolor='rgba(255,255,255,0.85)',
    )
    return style_figure(fig, 'Oil Volatility and Oil News Sentiment Through Time', height=600)


def make_correlation_heatmap():
    # Pearson correlation heatmap across all monthly airline IV, OVX mean, and TOSI series
    labels = ['AAL IV', 'DAL IV', 'UAL IV', 'LUV IV', 'JETS IV', 'OVX', 'TOSI']
    fig = go.Figure(go.Heatmap(
        z=monthly_corr.values, x=labels, y=labels,
        zmid=0, zmin=-1, zmax=1, colorscale='RdBu',
        text=np.round(monthly_corr.values, 2), texttemplate='%{text}',
        hovertemplate='%{x} vs %{y}<br>Pearson r=%{z:.2f}<extra></extra>',
        colorbar=dict(title='Pearson r'),
    ))
    fig.add_annotation(
        x=0.99, y=1.12, xref='paper', yref='paper',
        text=f'Full monthly overlap: {full_overlap.shape[0]} months',
        showarrow=False, xanchor='right', bgcolor='rgba(255,255,255,0.85)',
    )
    return style_figure(fig, 'Monthly Correlation Structure for Airline Volatility, OVX, and TOSI', height=620)


def make_driver_scatter_figure(driver_col, degree, title):
    # 2x3 scatter grid: one panel per ticker showing driver vs airline IV
    # with a polynomial fit curve and an in-panel correlation/R² annotation
    positions = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2)]
    fig = make_subplots(rows=2, cols=3, subplot_titles=TICKERS + [''])

    for subplot_number, ticker in enumerate(TICKERS, start=1):
        row, col = positions[subplot_number - 1]
        vol_col  = f'{ticker}_Volatility'
        pair = monthly_panel[[driver_col, vol_col]].dropna()
        x, y = pair[driver_col].to_numpy(), pair[vol_col].to_numpy()

        # Fit polynomial and generate a smooth curve for plotting
        coeffs = np.polyfit(x, y, degree)
        poly   = np.poly1d(coeffs)
        x_line = np.linspace(x.min(), x.max(), 200)
        y_line = poly(x_line)
        fit_stats  = polynomial_fit_scores(x, y)
        corr_value = pair[driver_col].corr(pair[vol_col])

        fig.add_trace(go.Scatter(
            x=x, y=y, mode='markers', name=ticker,
            legendgroup=ticker, showlegend=False,
            marker=dict(color=AIRLINE_COLORS[ticker], size=8, opacity=0.78),
            hovertemplate=f'{ticker}<br>{driver_col}=%{{x:.2f}}<br>IV=%{{y:.2f}}<extra></extra>',
        ), row=row, col=col)
        fig.add_trace(go.Scatter(
            x=x_line, y=y_line, mode='lines', name=f'{ticker} fit',
            legendgroup=ticker, showlegend=False,
            line=dict(color='#24323D', width=2), hoverinfo='skip',
        ), row=row, col=col)

        # In-panel annotation showing Pearson r and cubic R²
        suffix = axis_suffix(subplot_number)
        fig.add_annotation(
            x=0.98, y=0.96, xanchor='right', yanchor='top',
            xref=f'x{suffix} domain', yref=f'y{suffix} domain',
            text=(
                f'r = {corr_value:.2f}<br>'
                f"Best fit: {fit_stats['Best_Fit']}<br>"
                f"Cubic R² = {fit_stats['R2_Cubic']:.2f}"
            ),
            showarrow=False, align='right', bgcolor='rgba(255,255,255,0.82)',
        )

    fig.update_xaxes(title=driver_col)
    fig.update_yaxes(title='Monthly Mean Implied Volatility (%)')
    fig.update_layout(showlegend=False)
    return style_figure(fig, title, height=760, legend_y=1.02)

In [6]:

def make_model_comparison_figure():
    # Grouped bar chart: out-of-sample RMSE for every model/specification combo
    # across all five tickers; hover reveals MAE, MAPE, directional accuracy, and R²
    order = [
        'MIDAS | OVX Only', 'MIDAS | TOSI Only', 'MIDAS | OVX + TOSI',
        'XGBoost | OVX Only', 'XGBoost | TOSI Only', 'XGBoost | OVX + TOSI',
        'Random Forest | OVX Only', 'Random Forest | TOSI Only', 'Random Forest | OVX + TOSI',
    ]
    fig = go.Figure()
    for model_name in order:
        subset = model_comparison_df[model_comparison_df['Model'] == model_name].set_index('Ticker').reindex(TICKERS)
        fig.add_trace(go.Bar(
            x=TICKERS,
            y=subset['RMSE'],
            name=model_name,
            marker_color=MODEL_COLORS[model_name],
            customdata=np.stack([
                subset['MAE'].to_numpy(),
                subset['MAPE'].to_numpy(),
                subset['Directional_Accuracy'].to_numpy(),
                subset['R2'].to_numpy(),
            ], axis=1),
            hovertemplate=(
                'Ticker=%{x}<br>'
                'Model=%{fullData.name}<br>'
                'RMSE=%{y:.2f}<br>'
                'MAE=%{customdata[0]:.2f}<br>'
                'MAPE=%{customdata[1]:.1f}%<br>'
                'Directional accuracy=%{customdata[2]:.1%}<br>'
                'R?=%{customdata[3]:.2f}<extra></extra>'
            ),
        ))

    fig.update_xaxes(title='Ticker')
    fig.update_yaxes(title='Out-of-sample RMSE')
    fig.update_layout(barmode='group')
    fig = style_figure(fig, 'Next-Month Volatility Forecast Accuracy by Model Specification and Ticker', height=800)
    fig.update_layout(
        legend=dict(orientation='v', x=1.02, y=0.5, xanchor='left', yanchor='middle', font=dict(size=10)),
        margin=dict(l=60, r=250, t=80, b=60),
    )
    return fig


def make_forecast_comparison_figure():
    # 3x2 subplot grid: for each ticker, plot actual vs predicted next-month IV
    # from MIDAS, XGBoost, and Random Forest (all using the OVX + TOSI specification)
    positions      = [(1, 1), (1, 2), (2, 1), (2, 2), (3, 1)]
    subplot_titles = [f'{ticker} forecast panel' for ticker in TICKERS] + ['']
    fig = make_subplots(rows=3, cols=2, subplot_titles=subplot_titles, vertical_spacing=0.12)

    for idx, ticker in enumerate(TICKERS):
        row, col = positions[idx]
        # Align ML test-set predictions with the MIDAS rolling predictions on target_date
        merged = ml_predictions[ticker].merge(
            midas_predictions[ticker][['target_date', 'predicted']].rename(columns={'predicted': 'MIDAS | OVX + TOSI'}),
            on='target_date', how='left',
        )
        # Actual volatility line
        fig.add_trace(go.Scatter(
            x=merged['target_date'], y=merged['actual'],
            mode='lines+markers', name='Actual',
            legendgroup='Actual', showlegend=(idx == 0),
            line=dict(color='#24323D', width=2.5),
            hovertemplate='Actual<br>%{x|%Y-%m}<br>%{y:.2f}<extra></extra>',
        ), row=row, col=col)

        # One line per model (dashed for MIDAS, solid for tree-based models)
        for model_name, color in [('MIDAS | OVX + TOSI', '#2D3047'), ('XGBoost', '#1B998B'), ('Random Forest', '#FF6B35')]:
            if model_name not in merged:
                continue
            fig.add_trace(go.Scatter(
                x=merged['target_date'], y=merged[model_name],
                mode='lines', name=model_name,
                legendgroup=model_name, showlegend=(idx == 0),
                line=dict(color=color, width=2, dash='dash' if 'MIDAS' in model_name else 'solid'),
                hovertemplate=f'{model_name}<br>%{{x|%Y-%m}}<br>%{{y:.2f}}<extra></extra>',
            ), row=row, col=col)

        # Annotate each panel with the best-performing model for that ticker
        best_model = best_models_df.loc[best_models_df['Ticker'] == ticker, 'Model'].iloc[0]
        fig.add_annotation(
            x=0.02, y=0.04, yanchor='bottom',
            xref=f'x{axis_suffix(idx + 1)} domain', yref=f'y{axis_suffix(idx + 1)} domain',
            text=f'Best: {best_model}', showarrow=False,
            font=dict(size=9), bgcolor='rgba(255,255,255,0.82)',
        )

    fig.update_xaxes(title='Forecast month')
    fig.update_yaxes(title='Average implied volatility (%)')
    fig = style_figure(fig, 'Actual vs Predicted Next-Month Airline Volatility (Combined Specification)', height=1050)
    fig.update_layout(margin=dict(l=60, r=30, t=80, b=130))
    return fig


def make_midas_horizon_figure():
    # Side-by-side bars: MIDAS RMSE (left) and directional accuracy (right)
    # across 1-, 3-, and 6-month OVX windows (JETS only)
    fig = make_subplots(rows=1, cols=2, subplot_titles=['RMSE', 'Directional Accuracy'], horizontal_spacing=0.18)
    for spec in ['OVX Only', 'TOSI Only', 'OVX + TOSI']:
        subset = midas_horizon_df[midas_horizon_df['Specification'] == spec].sort_values('Horizon_Months')
        fig.add_trace(go.Bar(
            x=subset['Horizon_Label'], y=subset['RMSE'],
            name=spec, marker_color=MODEL_COLORS[spec],
            hovertemplate='Horizon=%{x}<br>RMSE=%{y:.2f}<extra></extra>',
            showlegend=True,
        ), row=1, col=1)
        fig.add_trace(go.Bar(
            x=subset['Horizon_Label'], y=subset['Directional_Accuracy'],
            name=spec, marker_color=MODEL_COLORS[spec],
            hovertemplate='Horizon=%{x}<br>Directional accuracy=%{y:.1%}<extra></extra>',
            showlegend=False,
        ), row=1, col=2)

    fig.add_hline(y=0.50, line_dash='dash', line_color='#7F8C8D', row=1, col=2)  # 50% random-chance baseline
    fig.update_yaxes(title='RMSE', row=1, col=1)
    fig.update_yaxes(title='Directional accuracy', tickformat='.0%', row=1, col=2)
    fig = style_figure(fig, 'JETS MIDAS Forecast Quality Across 1-, 3-, and 6-Month Horizons', height=600)
    fig.update_layout(margin=dict(l=60, r=30, t=100, b=130))
    return fig


def make_feature_importance_figure(ticker='JETS', top_n=10):
    # Horizontal bar charts of top-N feature importances for XGBoost and Random Forest
    # (OVX + TOSI specification) for the given ticker
    fig = make_subplots(rows=1, cols=2, subplot_titles=['XGBoost', 'Random Forest'], horizontal_spacing=0.32)
    for col_index, model_name in enumerate(['XGBoost', 'Random Forest'], start=1):
        importance = feature_importances[ticker][model_name].head(top_n).sort_values(ascending=True)
        fig.add_trace(go.Bar(
            x=importance.values, y=importance.index,
            orientation='h',
            marker_color='#1B998B' if model_name == 'XGBoost' else '#FF6B35',
            name=model_name, showlegend=False,
            hovertemplate='%{y}<br>Importance=%{x:.3f}<extra></extra>',
        ), row=1, col=col_index)

    fig.update_xaxes(title='Feature importance')
    fig = style_figure(fig, f'Top {top_n} Drivers of JETS Next-Month Volatility Forecasts', height=620)
    fig.update_layout(margin=dict(l=200, r=30, t=80, b=55))
    return fig


# --- Build all nine figures ---
fig_airline_timeseries = make_airline_timeseries_figure()
fig_oil_drivers        = make_oil_driver_figure()
fig_correlation        = make_correlation_heatmap()
fig_ovx_scatter        = make_driver_scatter_figure('OVX_mean', 3, 'OVX vs Monthly Airline Volatility With Cubic Fits')
fig_tosi_scatter       = make_driver_scatter_figure('TOSI', 1, 'TOSI vs Monthly Airline Volatility With Linear Fits')
fig_model_comparison   = make_model_comparison_figure()
fig_forecasts          = make_forecast_comparison_figure()
fig_midas_horizons     = make_midas_horizon_figure()
fig_feature_importance = make_feature_importance_figure()

## 6. Results Summary

Display the main performance tables and render the final interactive figures for inspection.

In [7]:
print('Data inventory')
display(data_inventory)

print('Cleaning log')
display(cleaning_log)

print('Polynomial fit summary used to motivate non-linear models')
display(fit_summary.sort_values('Pair').reset_index(drop=True))

print('JETS MIDAS horizon comparison')
display(midas_horizon_df)

print('All-airline MIDAS results by specification')
display(midas_all_df)

print('Monthly XGBoost and Random Forest results by specification')
display(ml_results_df)

print('Average metrics by family and specification')
display(family_spec_summary_df)

print('Expanded model metrics snapshot')
display(metrics_snapshot_df)

print('Average metrics by model specification across airlines')
display(aggregate_metrics_df)

print('JETS time-series cross-validation summary')
display(cv_summary_df)


for fig in [
    fig_airline_timeseries,
    fig_oil_drivers,
    fig_correlation,
    fig_ovx_scatter,
    fig_tosi_scatter,
    fig_model_comparison,
    fig_forecasts,
    fig_midas_horizons,
    fig_feature_importance,
]:
    fig.show()


Data inventory


,Dataset,Rows,Columns,Start,End,Missing_Before_Cleaning
0,AAL IV,3089,1,2013-12-09,2026-02-27,13
1,DAL IV,4738,1,2007-05-01,2026-02-27,6
2,UAL IV,3875,1,2010-10-01,2026-02-27,1
3,LUV IV,11511,1,1980-07-28,2026-02-27,3533
4,JETS IV,2724,1,2015-04-30,2026-02-27,17
5,OVX,4905,1,2007-05-10,2026-02-25,174
6,TOSI,526,1,1982-01-01,2025-10-01,0


Cleaning log


,Dataset,Method,Rows_Removed,Missing_Before,Missing_After
0,AAL IV,Backfill then drop leading structural gaps,0,13,0
1,DAL IV,Backfill then drop leading structural gaps,1,6,0
2,UAL IV,Backfill then drop leading structural gaps,1,1,0
3,LUV IV,Backfill then drop leading structural gaps,1,3533,0
4,JETS IV,Backfill then drop leading structural gaps,1,17,0
5,OVX,Drop holiday rows with no published index value,174,174,0
6,TOSI,No cleaning needed,0,0,0


Polynomial fit summary used to motivate non-linear models


...

,Ticker,Family,Specification,Model,RMSE,MAE,MedAE,MAPE,sMAPE,Bias,Explained_Variance,R2,Directional_Accuracy
0,AAL,MIDAS,OVX Only,MIDAS | OVX Only,3.803,3.241,3.208,8.880,8.520,2.127,0.069,-0.355,0.581
1,AAL,Random Forest,TOSI Only,Random Forest | TOSI Only,4.101,3.201,2.787,8.833,8.493,0.926,-0.439,-0.516,0.607
2,AAL,Random Forest,OVX Only,Random Forest | OVX Only,4.275,3.282,2.585,9.102,8.803,0.218,-0.643,-0.648,0.607
3,AAL,Random Forest,OVX + TOSI,Random Forest | OVX + TOSI,4.598,3.587,2.831,9.929,9.506,0.687,-0.863,-0.906,0.571
4,AAL,MIDAS,OVX + TOSI,MIDAS | OVX + TOSI,5.024,4.372,3.917,11.842,11.116,3.270,-0.363,-1.365,0.628
5,AAL,MIDAS,TOSI Only,MIDAS | TOSI Only,6.106,5.126,4.613,13.949,12.821,4.130,-0.895,-2.493,0.605
6,AAL,XGBoost,OVX Only,XGBoost | OVX Only,6.233,5.434,4.652,15.053,14.233,2.215,-2.060,-2.502,0.393
7,AAL,XGBoost,TOSI Only,XGBoost | TOSI Only,6.413,4.792,3.559,13.101,12.358,1.510,-2.502,-2.707,0.500
8,AAL,XGBoost,OVX + TOSI,XGBoost | OVX + TOSI,7.066,5.216,4.080,14.445,13.300,2.384,-2.988,-3.501,0.536
9,DAL,XGBoost,TOSI Only,XGBoost | TOSI Only,6.073,4.529,3.050,11.164,11.382,-0.449,0.299,0.295,0.698


Average metrics by model specification across airlines


,Model,Avg_RMSE,Avg_MAE,Avg_MedAE,Avg_MAPE,Avg_sMAPE,Avg_Abs_Bias,Avg_Explained_Variance,Avg_R2,Avg_Directional_Accuracy
0,Random Forest | OVX Only,5.635,4.075,3.082,10.690,10.749,0.721,-0.236,-0.284,0.610
1,XGBoost | OVX Only,6.125,4.631,3.707,12.411,12.377,0.966,-0.608,-0.720,0.626
2,Random Forest | TOSI Only,6.164,4.555,3.531,11.879,11.718,1.033,-0.449,-0.513,0.573
3,Random Forest | OVX + TOSI,6.209,4.602,3.550,12.143,11.971,1.001,-0.471,-0.517,0.578
4,XGBoost | TOSI Only,6.636,4.764,3.383,12.518,12.585,0.996,-0.893,-1.002,0.566
5,MIDAS | OVX Only,6.950,5.524,5.034,14.462,13.723,3.387,-0.039,-0.423,0.554
6,XGBoost | OVX + TOSI,7.220,5.294,3.932,14.124,13.873,1.223,-1.173,-1.313,0.576
7,MIDAS | OVX + TOSI,7.248,5.777,5.059,15.073,14.227,3.607,-0.134,-0.656,0.570
8,MIDAS | TOSI Only,8.704,7.162,5.950,19.468,17.777,4.276,-0.550,-1.580,0.581


JETS time-series cross-validation summary


,Model,Specification,Fold_RMSE_Mean,Fold_RMSE_Std
0,Random Forest,OVX + TOSI,12.033,13.371
1,Random Forest,OVX Only,11.728,13.727
2,Random Forest,TOSI Only,12.152,13.676
3,XGBoost,OVX + TOSI,11.303,13.392
4,XGBoost,OVX Only,11.243,13.464
5,XGBoost,TOSI Only,12.097,13.697
